# Predicting Online Purchase Intention from Web Session Behaviour

This notebook investigates whether e-commerce browsing sessions can be classified as purchase or non-purchase sessions using supervised machine learning.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

## Initial data examination

In [2]:
df = pd.read_csv("../data/online_shoppers_intention.csv")
df.head(10)

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False
5,0,0.0,0,0.0,19,154.216667,0.015789,0.024561,0.0,0.0,Feb,2,2,1,3,Returning_Visitor,False,False
6,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
7,1,0.0,0,0.0,0,0.000000,0.200000,0.200000,0.0,0.0,Feb,1,2,1,5,Returning_Visitor,True,False
8,0,0.0,0,0.0,2,37.000000,0.000000,0.100000,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False
9,0,0.0,0,0.0,3,738.000000,0.000000,0.022222,0.0,0.4,Feb,2,4,1,2,Returning_Visitor,False,False


The first few rows show that each record represents one online shopping session. The target variable is `Revenue`, which shows whether the session ended in a purchase. In the rows displayed here, most sessions did not generate revenue, shown by `False` in the `Revenue` column.

The input features describe different parts of the user’s browsing session. These include page-count features, duration features, web analytics measures, timing features and visitor/device-related features. For example, the dataset records the number of `Administrative`, `Informational` and `ProductRelated` pages visited, the time spent on those page types, `BounceRates`, `ExitRates`, `PageValues`, closeness to a `SpecialDay`, `Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`, `VisitorType` and whether the session took place on a `Weekend`.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

The `df.info()` output shows that the dataset contains 12,330 rows and 18 columns. There are 17 input features and one target variable, `Revenue`. All columns have 12,330 non-null values, so there are no missing values that need to be handled at this stage.

The data contains mixed feature types, including integers, floats, strings and Boolean values. This means preprocessing will still be required before training most machine learning models. For example, `Month` and `VisitorType` are categorical text features, while `Weekend` and `Revenue` are Boolean values. Some integer columns, such as `OperatingSystems`, `Browser`, `Region` and `TrafficType`, appear to be coded categories rather than true continuous numerical measurements, so they should be treated carefully during preprocessing, so that for example, browser 4 is not treated as “bigger” or “better” than browser 2.


In [4]:
df.shape

(12330, 18)

The dataset shape confirms that there are 12,330 rows and 18 columns. Each row represents one shopping session, and the columns include 17 input features plus the target variable, `Revenue`.

This is a suitable size for a standard supervised classification task. Because the dataset is not extremely small, I can use a normal train-test split later, and may also use cross-validation during model comparison or tuning.

In [5]:
df.isnull().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

The missing value check shows that every column has zero missing values. This means no rows need to be removed and no imputation is required before modelling.

As a result, the next preprocessing steps will focus on converting categorical variables into a machine-learning format, handling Boolean values, and scaling numerical features where needed.

In [6]:
df.duplicated().sum()

np.int64(125)

The duplicate check shows that there are 125 exact duplicate rows. This is a small number compared with the full dataset of 12,330 rows.

In some datasets, duplicates may be errors and should be removed. However, this dataset represents web browsing sessions, so it is possible for different sessions to have identical recorded values. For this reason, I will not remove the duplicates automatically at this stage, but I will mention this as a potential data issue in the report.

In [7]:
df["Revenue"].value_counts()
df["Revenue"].value_counts(normalize=True) * 100

Revenue
False    84.525547
True     15.474453
Name: proportion, dtype: float64

In [8]:
df.dtypes

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object

The data types confirm that the dataset contains a mixture of numerical, categorical and Boolean features.

In [9]:
df.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


The summary statistics show that the numerical features have very different ranges. For example, page count and duration columns can have much larger values than rate based features such as `BounceRates` and `ExitRates`.

This suggests that scaling may be useful for models that are sensitive to feature magnitude, such as logistic regression or k-nearest neighbours. Tree-based models are less affected by scaling, but using a preprocessing pipeline will make the comparison between models more consistent.

In [10]:
categorical_cols = ["Month", "VisitorType", "Weekend", "OperatingSystems", "Browser", "Region", "TrafficType"]

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


Month
Month
May     3364
Nov     2998
Mar     1907
Dec     1727
Oct      549
Sep      448
Aug      433
Jul      432
June     288
Feb      184
Name: count, dtype: int64

VisitorType
VisitorType
Returning_Visitor    10551
New_Visitor           1694
Other                   85
Name: count, dtype: int64

Weekend
Weekend
False    9462
True     2868
Name: count, dtype: int64

OperatingSystems
OperatingSystems
2    6601
1    2585
3    2555
4     478
8      79
6      19
7       7
5       6
Name: count, dtype: int64

Browser
Browser
2     7961
1     2462
4      736
5      467
6      174
10     163
8      135
3      105
13      61
7       49
12      10
11       6
9        1
Name: count, dtype: int64

Region
Region
1    4780
3    2403
4    1182
2    1136
6     805
7     761
9     511
8     434
5     318
Name: count, dtype: int64

TrafficType
TrafficType
2     3913
1     2451
3     2052
4     1069
13     738
10     450
6      444
8      343
5      260
11     247
20     198
9       42
7       40
15

The categorical summaries show how the non-continuous features are distributed. Some categories appear much more often than others, which means the data is not evenly spread across all category values.

This matters because rare categories may be harder for a model to learn from. These features will need to be encoded before modelling, perhaps using one-hot encoding for categorical variables.

In [11]:
df.groupby("Revenue")[["PageValues", "BounceRates", "ExitRates", "ProductRelated", "ProductRelated_Duration"]].mean()

,PageValues,BounceRates,ExitRates,ProductRelated,ProductRelated_Duration
Revenue,,,,,
False,1.975998,0.025317,0.047378,28.714642,1069.987809
True,27.264518,0.005117,0.019555,48.210168,1876.209615


The grouped averages compare selected features between purchase and non-purchase sessions. This gives an early indication of which variables may help separate the two classes.

Features such as `PageValues`, `BounceRates`, `ExitRates`, `ProductRelated` and `ProductRelated_Duration` are useful to inspect because they relate directly to browsing behaviour. If purchase sessions show noticeably different averages, these variables may be important predictors later. I will check this more formally with feature importance or feature selection.